In [1]:
import pandas as pd
import numpy as np


In [4]:
unames = ["user_id", "gender", "age", "occupation", "zip"]
users = pd.read_table("../data/ml-1m/users.dat", sep="::",
                      header=None, names=unames, engine="python")

rnames = ["user_id", "movie_id", "rating", "timestamp"]
ratings = pd.read_table("../data/ml-1m/ratings.dat", sep="::",
                        header=None, names=rnames, engine="python")

mnames = ["movie_id", "title", "genres"]
movies = pd.read_csv("../data/ml-1m/movies.dat", sep="::",
                      header=None, names=mnames, encoding="latin-1", engine="python")
# note the encoding. Different from utf-8

In [5]:
users.head(5)

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [6]:
movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


### Time to merge the tables to form one dataframe

In [8]:
moviesdf = pd.merge(pd.merge(ratings, users), movies)

In [9]:
moviesdf

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama


In [20]:
# Check the first row
moviesdf.iloc[0]

user_id                                            1
movie_id                                        1193
rating                                             5
timestamp                                  978300760
gender                                             F
age                                                1
occupation                                        10
zip                                            48067
title         One Flew Over the Cuckoo's Nest (1975)
genres                                         Drama
Name: 0, dtype: object

#### Mean movie ratings by gender

In [10]:
mean_ratings = moviesdf.pivot_table("rating", columns="gender", index="title", aggfunc="mean")
mean_ratings

gender,F,M
title,,
"$1,000,000 Duck (1971)",3.375000,2.761905
'Night Mother (1986),3.388889,3.352941
'Til There Was You (1997),2.675676,2.733333
"'burbs, The (1989)",2.793478,2.962085
...And Justice for All (1979),3.828571,3.689024
...,...,...
"Zed & Two Noughts, A (1985)",3.500000,3.380952
Zero Effect (1998),3.864407,3.723140
Zero Kelvin (Kjærlighetens kjøtere) (1995),NaN,3.500000


#### Which movies have at least 250 ratings

In [27]:
ratings_title = moviesdf.groupby("title").size()
ratings_title

title
$1,000,000 Duck (1971)                         37
'Night Mother (1986)                           70
'Til There Was You (1997)                      52
'burbs, The (1989)                            303
...And Justice for All (1979)                 199
                                             ... 
Zed & Two Noughts, A (1985)                    29
Zero Effect (1998)                            301
Zero Kelvin (Kjærlighetens kjøtere) (1995)      2
Zeus and Roxanne (1997)                        23
eXistenZ (1999)                               410
Length: 3706, dtype: int64

In [34]:
active_titles = ratings_title.index[ratings_title >= 250]
active_titles

Index([''burbs, The (1989)', '10 Things I Hate About You (1999)',
       '101 Dalmatians (1961)', '101 Dalmatians (1996)', '12 Angry Men (1957)',
       '13th Warrior, The (1999)', '2 Days in the Valley (1996)',
       '20,000 Leagues Under the Sea (1954)', '2001: A Space Odyssey (1968)',
       '2010 (1984)',
       ...
       'X-Men (2000)', 'Year of Living Dangerously (1982)',
       'Yellow Submarine (1968)', 'You've Got Mail (1998)',
       'Young Frankenstein (1974)', 'Young Guns (1988)',
       'Young Guns II (1990)', 'Young Sherlock Holmes (1985)',
       'Zero Effect (1998)', 'eXistenZ (1999)'],
      dtype='object', name='title', length=1216)

In [42]:
# use the active titles to get their index from mean_ratings dataframe
mean_ratings = mean_ratings.loc[active_titles]

In [47]:
# What movies did women rate the most?
mean_ratings["F"].sort_values(ascending=False)

title
Close Shave, A (1995)                                     4.644444
Wrong Trousers, The (1993)                                4.588235
Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)             4.572650
Wallace & Gromit: The Best of Aardman Animation (1996)    4.563107
Schindler's List (1993)                                   4.562602
                                                            ...   
Avengers, The (1998)                                      1.915254
Speed 2: Cruise Control (1997)                            1.906667
Rocky V (1990)                                            1.878788
Barb Wire (1996)                                          1.585366
Battlefield Earth (2000)                                  1.574468
Name: F, Length: 1216, dtype: float64

In [48]:
# To see the corresponding male ratings
mean_ratings.sort_values("F", ascending=False)

gender,F,M
title,,
"Close Shave, A (1995)",4.644444,4.473795
"Wrong Trousers, The (1993)",4.588235,4.478261
Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),4.572650,4.464589
Wallace & Gromit: The Best of Aardman Animation (1996),4.563107,4.385075
Schindler's List (1993),4.562602,4.491415
...,...,...
"Avengers, The (1998)",1.915254,2.017467
Speed 2: Cruise Control (1997),1.906667,1.863014
Rocky V (1990),1.878788,2.132780


## Rating disagreement

The higher the difference, the stronger the division

In [49]:
mean_ratings["diff"] = mean_ratings["M"] - mean_ratings["F"] 

In [50]:
mean_ratings.head()

gender,F,M,diff
title,,,
"'burbs, The (1989)",2.793478,2.962085,0.168607
10 Things I Hate About You (1999),3.646552,3.311966,-0.334586
101 Dalmatians (1961),3.791444,3.500000,-0.291444
101 Dalmatians (1996),3.240000,2.911215,-0.328785
12 Angry Men (1957),4.184397,4.328421,0.144024


A negative `diff` means that women rated it better than men. The wider the `diff` the more polar the disagreement.
    We'll now see what movies divided the party by sorting on the `diff` column.

In [51]:
sort_diff = mean_ratings.sort_values("diff")

In [53]:
# The ones preferred by women
sort_diff.head(10)

gender,F,M,diff
title,,,
Dirty Dancing (1987),3.790378,2.959596,-0.830782
Jumpin' Jack Flash (1986),3.254717,2.578358,-0.676359
Grease (1978),3.975265,3.367041,-0.608224
Little Women (1994),3.870588,3.321739,-0.548849
Steel Magnolias (1989),3.901734,3.365957,-0.535777
Anastasia (1997),3.800000,3.281609,-0.518391
"Rocky Horror Picture Show, The (1975)",3.673016,3.160131,-0.512885
"Color Purple, The (1985)",4.158192,3.659341,-0.498851
"Age of Innocence, The (1993)",3.827068,3.339506,-0.487561


In [57]:
# the ones preferred by men
sort_diff.loc[::-1].head(10)

gender,F,M,diff
title,,,
"Good, The Bad and The Ugly, The (1966)",3.494949,4.221300,0.726351
"Kentucky Fried Movie, The (1977)",2.878788,3.555147,0.676359
Dumb & Dumber (1994),2.697987,3.336595,0.638608
"Longest Day, The (1962)",3.411765,4.031447,0.619682
"Cable Guy, The (1996)",2.250000,2.863787,0.613787
Evil Dead II (Dead By Dawn) (1987),3.297297,3.909283,0.611985
"Hidden, The (1987)",3.137931,3.745098,0.607167
Rocky III (1982),2.361702,2.943503,0.581801
Caddyshack (1980),3.396135,3.969737,0.573602


### Which ones polarized everyone? 
We should see this through the variance (standard deviation).

In [61]:
all_rating_std = moviesdf.groupby("title")["rating"].std()

In [62]:
active_titles_std = all_rating_std.loc[active_titles]

In [64]:
active_titles_std.head(10)

title
'burbs, The (1989)                     1.107760
10 Things I Hate About You (1999)      0.989815
101 Dalmatians (1961)                  0.982103
101 Dalmatians (1996)                  1.098717
12 Angry Men (1957)                    0.812731
13th Warrior, The (1999)               1.140421
2 Days in the Valley (1996)            0.921592
20,000 Leagues Under the Sea (1954)    0.869685
2001: A Space Odyssey (1968)           1.042504
2010 (1984)                            0.946618
Name: rating, dtype: float64

In [67]:
# Which are the 10 most polarising movies in here?

active_titles_std.sort_values(ascending=False).head(10)

title
Dumb & Dumber (1994)                     1.321333
Blair Witch Project, The (1999)          1.316368
Natural Born Killers (1994)              1.307198
Tank Girl (1995)                         1.277695
Rocky Horror Picture Show, The (1975)    1.260177
Eyes Wide Shut (1999)                    1.259624
Evita (1996)                             1.253631
Billy Madison (1995)                     1.249970
Fear and Loathing in Las Vegas (1998)    1.246408
Bicentennial Man (1999)                  1.245533
Name: rating, dtype: float64

In [68]:
type(active_titles_std)

pandas.core.series.Series

### Let's look at what genres the ratings belong to

In [70]:
moviesdf["genres"].head(10)

0                                 Drama
1          Animation|Children's|Musical
2                       Musical|Romance
3                                 Drama
4           Animation|Children's|Comedy
5       Action|Adventure|Comedy|Romance
6                Action|Adventure|Drama
7                          Comedy|Drama
8          Animation|Children's|Musical
9    Adventure|Children's|Drama|Musical
Name: genres, dtype: object

In [11]:
# Make a list of the genres for each movie. Use the split() method
moviesdf["genre"] = moviesdf.pop("genres").str.split("|")

In [12]:
moviesdf.head()

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genre
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),[Drama]
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),"[Animation, Children's, Musical]"
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),"[Musical, Romance]"
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),[Drama]
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)","[Animation, Children's, Comedy]"


In [13]:
# Call explode() on the dataframe and column of interest ("genre") to create a row fo each extra genre in the list.

moviesdf_exploded = moviesdf.explode("genre")

In [14]:
moviesdf_exploded.head(10)

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genre
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Children's
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Children's
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Comedy


In [15]:
# New dataframe using the ratings and the exploded genre and users.
movies["genre"] = movies.pop("genres").str.split("|")
movies_exploded = movies.explode("genre")
movies_exploded

,movie_id,title,genre
0,1,Toy Story (1995),Animation
0,1,Toy Story (1995),Children's
0,1,Toy Story (1995),Comedy
1,2,Jumanji (1995),Adventure
1,2,Jumanji (1995),Children's
...,...,...,...
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama
3882,3952,"Contender, The (2000)",Drama


In [16]:
moviesdf_exploded

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genre
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Children's
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical
...,...,...,...,...,...,...,...,...,...,...
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama
1000208,6040,1097,4,956715569,M,25,6,11106,E.T. the Extra-Terrestrial (1982),Children's
1000208,6040,1097,4,956715569,M,25,6,11106,E.T. the Extra-Terrestrial (1982),Drama
1000208,6040,1097,4,956715569,M,25,6,11106,E.T. the Extra-Terrestrial (1982),Fantasy


In [17]:
ratings_genre = pd.merge(pd.merge(movies_exploded, ratings), users)

In [18]:
ratings_genre

,movie_id,title,genre,user_id,rating,timestamp,gender,age,occupation,zip
0,1,Toy Story (1995),Animation,1,5,978824268,F,1,10,48067
1,1,Toy Story (1995),Animation,6,4,978237008,F,50,9,55117
2,1,Toy Story (1995),Animation,8,4,978233496,M,25,12,11413
3,1,Toy Story (1995),Animation,9,5,978225952,M,25,17,61614
4,1,Toy Story (1995),Animation,10,5,978226474,F,35,1,95370
...,...,...,...,...,...,...,...,...,...,...
2101810,3952,"Contender, The (2000)",Thriller,5812,4,992072099,F,25,7,92120
2101811,3952,"Contender, The (2000)",Thriller,5831,3,986223125,M,25,1,92120
2101812,3952,"Contender, The (2000)",Thriller,5837,4,1011902656,M,25,7,60607
2101813,3952,"Contender, The (2000)",Thriller,5927,1,979852537,M,35,14,10003


In [23]:
genre_ratings = ratings_genre.groupby(["genre", "age"])["rating"].mean().unstack("age")

In [24]:
genre_ratings

age,1,18,25,35,45,50,56
genre,,,,,,,
Action,3.506385,3.447097,3.453358,3.538107,3.528543,3.611333,3.610709
Adventure,3.449975,3.408525,3.443163,3.515291,3.528963,3.628163,3.649064
Animation,3.476113,3.624014,3.701228,3.740545,3.734856,3.780020,3.756233
Children's,3.241642,3.294257,3.426873,3.518423,3.527593,3.556555,3.621822
Comedy,3.497491,3.460417,3.490385,3.561984,3.591789,3.646868,3.650949
Crime,3.710170,3.668054,3.680321,3.733736,3.750661,3.810688,3.832549
Documentary,3.730769,3.865865,3.946690,3.953747,3.966521,3.908108,3.961538
Drama,3.794735,3.721930,3.726428,3.782512,3.784356,3.878415,3.933465
Fantasy,3.317647,3.353778,3.452484,3.482301,3.532468,3.581570,3.532700
